#Environment & Workspace Setup

In [ ]:
import os

# Mount Drive if running interactively in Colab with persistent storage, otherwise fall back to local runtime storage
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/PEFT_Comparison"
except Exception:
    PROJECT_DIR = "./peft_project"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f"Working directory set to: {os.getcwd()}")

Mounted at /content/drive
Working directory set to: /content/drive/MyDrive/PEFT_Comparison


Installing and Importing dependencies

In [ ]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate evaluate scikit-learn matplotlib seaborn
!pip uninstall -y torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 80.7 MB/s eta 0:00:00


In [ ]:
#Verify GPU and CUDA Availability

import torch
assert torch.cuda.is_available(), "CUDA GPU is not enabled! Switch runtime to T4 GPU."
device_name = torch.cuda.get_device_name(0)
total_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f"Device: {device_name}")
print(f"Total VRAM: {total_memory:.2f} GB")

Device: Tesla T4
Total VRAM: 14.56 GB


#Data Pipeline & Preprocessing

In [ ]:
%%writefile data.py
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

def prepare_imdb_data(
    model_name: str = "distilbert-base-uncased",
    train_size: int = 5000,
    val_size: int = 1000,
    test_size: int = 1000,
    max_length: int = 256,
    seed: int = 42
):
    """
    Loads IMDB dataset, creates reproducible train/val/test splits,
    tokenizes, and formats tensors for PyTorch.
    """
    print("Loading raw IMDB dataset (stanfordnlp/imdb)...")
    raw_datasets = load_dataset("stanfordnlp/imdb")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    full_train = raw_datasets["train"].shuffle(seed=seed)
    train_subset = full_train.select(range(train_size))

    train_val_split = train_subset.train_test_split(test_size=val_size, seed=seed)
    train_data = train_val_split["train"]
    val_data = train_val_split["test"]

    test_subset = raw_datasets["test"].shuffle(seed=seed).select(range(test_size))

    print(f"Splits created: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_subset)}")

    def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

    print("Tokenizing datasets...")
    tokenized_train = train_data.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized_val = val_data.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized_test = test_subset.map(tokenize_fn, batched=True, remove_columns=["text"])

    tokenized_train.set_format("torch")
    tokenized_val.set_format("torch")
    tokenized_test.set_format("torch")

    return {
        "train": tokenized_train,
        "val": tokenized_val,
        "test": tokenized_test,
        "tokenizer": tokenizer
    }

if __name__ == "__main__":
    splits = prepare_imdb_data()
    print("Data preparation complete and verified.")

Overwriting data.py


In [ ]:
#Execute Preprocessing & Cache Splits to Google Drive

import importlib
import data
importlib.reload(data)
from data import prepare_imdb_data

DATA_CACHE_DIR = os.path.join(PROJECT_DIR, "data_cache")

data_dict = prepare_imdb_data(
    model_name="distilbert-base-uncased",
    train_size=5000,
    val_size=1000,
    test_size=1000,
    max_length=256,
    seed=42
)

os.makedirs(DATA_CACHE_DIR, exist_ok=True)
data_dict["train"].save_to_disk(os.path.join(DATA_CACHE_DIR, "train"))
data_dict["val"].save_to_disk(os.path.join(DATA_CACHE_DIR, "val"))
data_dict["test"].save_to_disk(os.path.join(DATA_CACHE_DIR, "test"))
data_dict["tokenizer"].save_pretrained(os.path.join(DATA_CACHE_DIR, "tokenizer"))

print(f"Tokenized datasets successfully saved to {DATA_CACHE_DIR}")

Loading raw IMDB dataset (stanfordnlp/imdb)...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenized datasets successfully saved to /content/drive/MyDrive/PEFT_Comparison/data_cache


In [ ]:
#Verification Check

sample = data_dict["train"][0]
print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["label"].item())
labels = [x["label"].item() for x in data_dict["train"]]
pos_count = sum(labels)
neg_count = len(labels) - pos_count
print(f"Class balance in Train: Positive={pos_count}, Negative={neg_count}")

input_ids shape: torch.Size([256])
attention_mask shape: torch.Size([256])
label: 0
Class balance in Train: Positive=2012, Negative=1988


#Full Fine-Tuning Baseline

In [ ]:
%%writefile utils.py
import os
import time
import json
import torch
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    """Computes Accuracy and F1 Score."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

def get_parameter_stats(model):
    """Calculates total and trainable parameters."""
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    pct = 100 * trainable_params / total_params
    return trainable_params, total_params, pct

class PerformanceTracker:
    """Tracks training time and peak GPU memory."""
    def __init__(self):
        self.start_time = None
        self.end_time = None
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.empty_cache()

    def start(self):
        self.start_time = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def stop(self):
        self.end_time = time.time()
        peak_bytes = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0
        peak_mb = peak_bytes / (1024 ** 2)
        duration_sec = self.end_time - self.start_time
        return duration_sec, peak_mb

def save_experiment_results(results_dict, output_path):
    """Saves metrics to JSON for Phase 6 aggregation."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(results_dict, f, indent=4)
    print(f"Metrics saved to {output_path}")

Writing utils.py


In [ ]:
%%writefile train_full.py
import os
import torch
from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from utils import compute_metrics, get_parameter_stats, PerformanceTracker, save_experiment_results

def run_full_finetuning(
    data_dir="data_cache",
    output_dir="results/full_ft",
    epochs=3,
    batch_size=16,
    lr=2e-5,
    seed=42
):
    set_seed(seed)
    print("\n--- Starting Full Fine-Tuning Pipeline ---")

    train_data = load_from_disk(os.path.join(data_dir, "train"))
    val_data = load_from_disk(os.path.join(data_dir, "val"))
    test_data = load_from_disk(os.path.join(data_dir, "test"))

    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2
    )

    trainable_p, total_p, pct = get_parameter_stats(model)
    print(f"Total Params: {total_p:,} | Trainable: {trainable_p:,} ({pct:.2f}%)")

    training_args = TrainingArguments(
        output_dir=os.path.join(output_dir, "checkpoints"),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        seed=seed,
        fp16=torch.cuda.is_available(),  # Fast standard mixed precision on T4
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=val_data,
        compute_metrics=compute_metrics
    )

    tracker = PerformanceTracker()
    tracker.start()
    train_result = trainer.train()
    duration_sec, peak_mem_mb = tracker.stop()

    print(f"\nTraining Complete!")
    print(f"Wall-clock Time: {duration_sec:.2f}s ({duration_sec / 60:.2f} min)")
    print(f"Peak GPU VRAM: {peak_mem_mb:.2f} MB")

    print("\nEvaluating on Test Split...")
    test_metrics = trainer.evaluate(test_data)
    print(f"Test Accuracy: {test_metrics['eval_accuracy']:.4f}")
    print(f"Test F1: {test_metrics['eval_f1']:.4f}")

    loss_history = [
        {"step": log["step"], "loss": log["loss"]}
        for log in trainer.state.log_history
        if "loss" in log
    ]

    results = {
        "technique": "Full Fine-Tuning",
        "total_params": total_p,
        "trainable_params": trainable_p,
        "trainable_percent": pct,
        "training_time_sec": duration_sec,
        "peak_memory_mb": peak_mem_mb,
        "test_accuracy": test_metrics["eval_accuracy"],
        "test_f1": test_metrics["eval_f1"],
        "loss_history": loss_history
    }

    save_experiment_results(results, os.path.join(output_dir, "metrics.json"))
    return results

if __name__ == "__main__":
    run_full_finetuning()

Writing train_full.py


Execute Full Fine-Tuning

In [ ]:
!python train_full.py


--- Starting Full Fine-Tuning Pipeline ---
Loading weights: 100% 100/100 [00:00<00:00, 4149.69it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Total Params: 66,955,010 | Trainable: 66,955,010 (100.00%)
{'loss': '0.6623', 'grad_norm': '5.708', 'learning_rate': '

#LoRA Fine-Tuning

In [ ]:
%%writefile train_lora.py
import os
import torch
from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from peft import LoraConfig, get_peft_model, TaskType
from utils import compute_metrics, get_parameter_stats, PerformanceTracker, save_experiment_results

def run_lora_finetuning(
    data_dir="data_cache",
    output_dir="results/lora",
    epochs=3,
    batch_size=16,
    lr=5e-4,  # LoRA requires a higher learning rate (1e-4 to 5e-4) than full fine-tuning
    seed=42
):
    set_seed(seed)
    print("\n--- Starting LoRA Fine-Tuning Pipeline ---")

    train_data = load_from_disk(os.path.join(data_dir, "train"))
    val_data = load_from_disk(os.path.join(data_dir, "val"))
    test_data = load_from_disk(os.path.join(data_dir, "test"))

    base_model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2
    )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["q_lin", "v_lin"],
        bias="none"
    )

    model = get_peft_model(base_model, lora_config)
    trainable_p, total_p, pct = get_parameter_stats(model)
    print(f"Total Params: {total_p:,} | Trainable: {trainable_p:,} ({pct:.2f}%)")

    training_args = TrainingArguments(
        output_dir=os.path.join(output_dir, "checkpoints"),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        seed=seed,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=val_data,
        compute_metrics=compute_metrics
    )

    tracker = PerformanceTracker()
    tracker.start()
    train_result = trainer.train()
    duration_sec, peak_mem_mb = tracker.stop()

    print(f"\nTraining Complete!")
    print(f"Wall-clock Time: {duration_sec:.2f}s ({duration_sec / 60:.2f} min)")
    print(f"Peak GPU VRAM: {peak_mem_mb:.2f} MB")

    print("\nEvaluating on Test Split...")
    test_metrics = trainer.evaluate(test_data)
    print(f"Test Accuracy: {test_metrics['eval_accuracy']:.4f}")
    print(f"Test F1: {test_metrics['eval_f1']:.4f}")

    loss_history = [
        {"step": log["step"], "loss": log["loss"]}
        for log in trainer.state.log_history
        if "loss" in log
    ]

    results = {
        "technique": "LoRA",
        "total_params": total_p,
        "trainable_params": trainable_p,
        "trainable_percent": pct,
        "training_time_sec": duration_sec,
        "peak_memory_mb": peak_mem_mb,
        "test_accuracy": test_metrics["eval_accuracy"],
        "test_f1": test_metrics["eval_f1"],
        "loss_history": loss_history
    }

    save_experiment_results(results, os.path.join(output_dir, "metrics.json"))
    model.save_pretrained(os.path.join(output_dir, "adapter_weights"))
    return results

if __name__ == "__main__":
    run_lora_finetuning()

Writing train_lora.py


Execute LoRA Fine-Tuning

In [ ]:
!python train_lora.py


--- Starting LoRA Fine-Tuning Pipeline ---
Loading weights: 100% 100/100 [00:00<00:00, 5144.68it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Total Params: 67,694,596 | Trainable: 739,586 (1.09%)
{'loss': '0.5691', 'grad_norm': '5.782', 'learning_rate': '0.000

#QLoRA Fine-Tuning

In [ ]:
%%writefile train_qlora.py
import os
import torch
from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    set_seed
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)
from utils import compute_metrics, get_parameter_stats, PerformanceTracker, save_experiment_results

def run_qlora_finetuning(
    data_dir="data_cache",
    output_dir="results/qlora",
    epochs=3,
    batch_size=16,
    lr=5e-4,
    seed=42
):
    set_seed(seed)
    print("\n--- Starting QLoRA (4-bit) Fine-Tuning Pipeline ---")

    train_data = load_from_disk(os.path.join(data_dir, "train"))
    val_data = load_from_disk(os.path.join(data_dir, "val"))
    test_data = load_from_disk(os.path.join(data_dir, "test"))

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
        # Keep classification heads unquantized in FP16/FP32 for stable training
        llm_int8_skip_modules=["pre_classifier", "classifier"]
    )
    # device_map={"": 0} ensures proper weight quantization initialization on GPU
    device_map = {"": 0} if torch.cuda.is_available() else None
    base_model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        quantization_config=bnb_config,
        device_map=device_map,
        num_labels=2
    )

    # use_gradient_checkpointing=False reduces computation overhead on small models
    base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=False)

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["q_lin", "v_lin"],
        bias="none"
    )

    model = get_peft_model(base_model, lora_config)
    trainable_p, total_p, pct = get_parameter_stats(model)
    print(f"Total Params: {total_p:,} | Trainable: {trainable_p:,} ({pct:.2f}%)")

    training_args = TrainingArguments(
        output_dir=os.path.join(output_dir, "checkpoints"),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        seed=seed,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=val_data,
        compute_metrics=compute_metrics
    )

    tracker = PerformanceTracker()
    tracker.start()
    train_result = trainer.train()
    duration_sec, peak_mem_mb = tracker.stop()

    print(f"\nTraining Complete!")
    print(f"Wall-clock Time: {duration_sec:.2f}s ({duration_sec / 60:.2f} min)")
    print(f"Peak GPU VRAM: {peak_mem_mb:.2f} MB")

    print("\nEvaluating on Test Split...")
    test_metrics = trainer.evaluate(test_data)
    print(f"Test Accuracy: {test_metrics['eval_accuracy']:.4f}")
    print(f"Test F1: {test_metrics['eval_f1']:.4f}")

    loss_history = [
        {"step": log["step"], "loss": log["loss"]}
        for log in trainer.state.log_history
        if "loss" in log
    ]

    results = {
        "technique": "QLoRA (4-bit)",
        "total_params": total_p,
        "trainable_params": trainable_p,
        "trainable_percent": pct,
        "training_time_sec": duration_sec,
        "peak_memory_mb": peak_mem_mb,
        "test_accuracy": test_metrics["eval_accuracy"],
        "test_f1": test_metrics["eval_f1"],
        "loss_history": loss_history
    }

    save_experiment_results(results, os.path.join(output_dir, "metrics.json"))
    model.save_pretrained(os.path.join(output_dir, "adapter_weights"))
    return results

if __name__ == "__main__":
    run_qlora_finetuning()

Overwriting train_qlora.py


Execute QLoRA Fine-Tuning

In [ ]:
!python train_qlora.py


--- Starting QLoRA (4-bit) Fine-Tuning Pipeline ---
Loading weights: 100% 100/100 [00:00<00:00, 421.97it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Total Params: 46,460,932 | Trainable: 739,586 (1.59%)
{'loss': '0.5594', 'grad_norm': '5.635', 'learning_rate'

#Sanity Verification & Results Aggregation (FR9)

In [ ]:
%%writefile evaluate.py
import os
import json
import pandas as pd

def aggregate_and_verify():
    paths = {
        "Full Fine-Tuning": "results/full_ft/metrics.json",
        "LoRA": "results/lora/metrics.json",
        "QLoRA (4-bit)": "results/qlora/metrics.json"
    }

    records = []
    for technique, path in paths.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing results for {technique} at {path}")
        with open(path, "r") as f:
            data = json.load(f)
            records.append({
                "Method": technique,
                "Trainable Params": f"{data['trainable_params']:,}",
                "Trainable %": f"{data['trainable_percent']:.2f}%",
                "Training Time (s)": round(data["training_time_sec"], 2),
                "Peak VRAM (MB)": round(data["peak_memory_mb"], 2),
                "Test Accuracy": round(data["test_accuracy"], 4),
                "Test F1": round(data["test_f1"], 4)
            })

    df = pd.DataFrame(records)

    os.makedirs("results", exist_ok=True)
    csv_path = "results/comparison_table.csv"
    md_path = "results/comparison_table.md"
    df.to_csv(csv_path, index=False)

    with open(md_path, "w") as f:
        f.write(df.to_markdown(index=False))

    print("\n--- Consolidated Comparison Table (FR9) ---")
    print(df.to_string(index=False))
    print(f"\nSaved CSV to: {csv_path}")
    print(f"Saved Markdown to: {md_path}")
    return df

if __name__ == "__main__":
    aggregate_and_verify()

Writing evaluate.py


In [ ]:
!python evaluate.py


--- Consolidated Comparison Table (FR9) ---
          Method Trainable Params Trainable %  Training Time (s)  Peak VRAM (MB)  Test Accuracy  Test F1
Full Fine-Tuning       66,955,010     100.00%             105.76         1720.34          0.886    0.886
            LoRA          739,586       1.09%              61.40          995.70          0.872    0.872
   QLoRA (4-bit)          739,586       1.59%              85.26          966.32          0.867    0.867

Saved CSV to: results/comparison_table.csv
Saved Markdown to: results/comparison_table.md


#Visualization (FR10)

In [ ]:
%%writefile visualize.py
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

os.makedirs("figures", exist_ok=True)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 11})

runs = {}
for name, p in [("Full FT", "results/full_ft/metrics.json"),
                ("LoRA", "results/lora/metrics.json"),
                ("QLoRA", "results/qlora/metrics.json")]:
    with open(p, "r") as f:
        runs[name] = json.load(f)

methods = list(runs.keys())
colors = ["#4C72B0", "#55A868", "#C44E52"]

plt.figure(figsize=(7, 4.5))
params = [runs[m]["trainable_params"] for m in methods]
bars = plt.bar(methods, params, color=colors, edgecolor='black', alpha=0.85)
plt.yscale("log")
plt.ylabel("Trainable Parameters (Log Scale)")
plt.title("Trainable Parameters by Fine-Tuning Strategy")
for bar, p in zip(bars, params):
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval * 1.3, f"{p:,}", ha='center', va='bottom', fontweight='bold')
plt.ylim(10**5, 10**8 * 2)
plt.tight_layout()
plt.savefig("figures/fig1_parameter_comparison.png", dpi=300)
plt.close()

plt.figure(figsize=(8, 4.5))
for idx, (m, color) in enumerate(zip(methods, colors)):
    history = runs[m]["loss_history"]
    steps = [x["step"] for x in history]
    losses = [x["loss"] for x in history]
    plt.plot(steps, losses, label=m, color=color, linewidth=2.2, marker='o' if idx > 0 else None, markersize=4)
plt.xlabel("Optimization Steps")
plt.ylabel("Training Loss")
plt.title("Training Loss Trajectory Across Steps")
plt.legend()
plt.tight_layout()
plt.savefig("figures/fig2_loss_curves.png", dpi=300)
plt.close()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
vram = [runs[m]["peak_memory_mb"] for m in methods]
ax1.bar(methods, vram, color=colors, edgecolor='black', alpha=0.85)
ax1.set_ylabel("Peak VRAM (MB)")
ax1.set_title("Peak GPU Memory Allocated")
for idx, val in enumerate(vram):
    ax1.text(idx, val + 35, f"{val:.1f} MB", ha='center', fontweight='bold')

times = [runs[m]["training_time_sec"] for m in methods]
ax2.bar(methods, times, color=colors, edgecolor='black', alpha=0.85)
ax2.set_ylabel("Wall-Clock Time (s)")
ax2.set_title("Total Training Duration")
for idx, val in enumerate(times):
    ax2.text(idx, val + 1.5, f"{val:.1f} s", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig("figures/fig3_hardware_efficiency.png", dpi=300)
plt.close()

plt.figure(figsize=(7, 5))
accs = [runs[m]["test_accuracy"] * 100 for m in methods]
for i, m in enumerate(methods):
    plt.scatter(vram[i], accs[i], color=colors[i], s=250, edgecolor='black', zorder=5)
    plt.annotate(
        f"{m}\n({accs[i]:.2f}%, {vram[i]:.0f}MB)",
        (vram[i], accs[i]),
        textcoords="offset points",
        xytext=(0, 12),
        ha='center',
        fontweight='bold'
    )
plt.xlabel("Peak GPU VRAM (MB)")
plt.ylabel("Test Accuracy (%)")
plt.title("Performance vs. Resource Consumption Trade-Off")
plt.xlim(min(vram) - 150, max(vram) + 200)
plt.ylim(min(accs) - 1.0, max(accs) + 1.5)
plt.tight_layout()
plt.savefig("figures/fig4_accuracy_vs_memory.png", dpi=300)
plt.close()

print("All 4 figures successfully generated and saved to 'figures/' directory.")

Writing visualize.py


In [ ]:
!python visualize.py

All 4 figures successfully generated and saved to 'figures/' directory.
